# Double Demeaning for Fixed Effects Interactions - Basic Example

This notebook demonstrates the use of the `dd_ie` package for analyzing interactions in fixed effects models using the double demeaning technique.

**Reference:** Giesselmann, M., & Schmidt-Catran, A. W. (2022). Interactions in Fixed Effects Regression Models. *Sociological Methods & Research*, 51(3), 1100-1127.

**Author:** Nikolaos Koutounidis


## Setup and Imports

First, let's import the necessary libraries and add the current directory to the Python path so we can import our modules directly.


In [1]:
import logging

import numpy as np
import pandas as pd

from dd_ie import DoubleDemeanAnalysis

# Configure logging so verbose output is visible in the notebook
logging.basicConfig(level=logging.INFO, format="%(levelname)s - %(message)s")

print(f"pandas {pd.__version__}, numpy {np.__version__}")
print("dd_ie imported successfully.")


pandas 2.2.3, numpy 2.4.1
dd_ie imported successfully.


## Generate Synthetic Panel Data

Let's create a realistic panel dataset that demonstrates the potential for bias in standard fixed effects interaction estimates. This data includes:

- **Unit-specific heterogeneity**: Each unit has its own interaction effect
- **Within-unit variation**: Both X and Z vary over time within units
- **Realistic structure**: 50 units × 8 time periods


In [2]:
def generate_synthetic_data(n_units=50, n_time=8, seed=42):
    """
    Generate synthetic panel data for demonstration.
    
    This creates a realistic panel dataset where:
    - Both X and Z vary within units
    - There's potential for unobserved effect heterogeneity
    - The true interaction effect differs from standard FE estimates
    """
    np.random.seed(seed)
    
    data = []
    
    for unit in range(1, n_units + 1):
        # Unit-specific parameters (unobserved heterogeneity)
        alpha_i = np.random.normal(0, 1)  # Unit fixed effect
        beta_interaction_i = np.random.normal(0.5, 0.2)  # Unit-specific interaction effect
        
        for time in range(1, n_time + 1):
            # Time-varying variables
            x_it = np.random.normal(unit/10, 1)  # Treatment variable with unit-specific trend
            z_it = np.random.normal(time/5, 1)   # Moderator variable with time trend
            control_it = np.random.normal(0, 1)  # Control variable
            
            # Generate outcome with unit-specific interaction effect
            # This creates the bias that double demeaning corrects for
            y_it = (alpha_i + 
                   0.8 * x_it + 
                   0.6 * z_it + 
                   beta_interaction_i * x_it * z_it +  # Unit-specific interaction
                   0.3 * control_it + 
                   np.random.normal(0, 0.5))  # Random error
            
            data.append({
                'unit_id': unit,
                'time_id': time,
                'outcome': y_it,
                'treatment': x_it,
                'moderator': z_it,
                'control': control_it
            })
    
    return pd.DataFrame(data)

# Generate the data
print("📊 Generating synthetic panel data...")
df = generate_synthetic_data(n_units=50, n_time=8)

print(f"✅ Created dataset with {len(df):,} observations")
print(f"   📈 Units: {df['unit_id'].nunique()}")
print(f"   📅 Time periods: {df['time_id'].nunique()}")
print(f"   📊 Variables: {list(df.columns)}")


📊 Generating synthetic panel data...
✅ Created dataset with 400 observations
   📈 Units: 50
   📅 Time periods: 8
   📊 Variables: ['unit_id', 'time_id', 'outcome', 'treatment', 'moderator', 'control']


## Explore the Data

Let's take a look at the structure and properties of our synthetic dataset.


In [3]:
# Display first few rows
print("📋 First 10 rows of the dataset:")
display(df.head(10))

# Summary statistics
print("\n📊 Summary statistics:")
display(df.describe())


📋 First 10 rows of the dataset:


,unit_id,time_id,outcome,treatment,moderator,control
0,1,1,2.549888,0.747689,1.723030,-0.234153
1,1,2,3.596959,1.679213,1.167435,-0.469474
2,1,3,-0.620558,-0.363418,0.134270,0.241962
3,1,4,-0.989769,-1.624918,0.237712,-1.012831
4,1,5,0.087082,-0.808024,-0.412304,1.465649
5,1,6,0.370250,0.167528,-0.224748,-0.544383
6,1,7,-0.486217,-1.050994,1.775698,-0.600639
7,1,8,0.815691,-0.501707,3.452278,-0.013497
8,2,1,-0.390253,0.408864,-1.759670,-1.328186
9,2,2,1.868072,0.938467,0.571368,-0.115648



📊 Summary statistics:


,unit_id,time_id,outcome,treatment,moderator,control
count,400.000000,400.000000,400.000000,400.000000,400.000000,400.000000
mean,25.500000,4.500000,3.616810,2.557174,0.859166,0.106163
std,14.448942,2.294157,3.433580,1.753068,1.090468,0.960648
min,1.000000,1.000000,-3.386093,-2.319745,-2.448543,-3.241267
25%,13.000000,2.750000,1.186356,1.249178,0.089275,-0.546654
50%,25.500000,4.500000,3.248646,2.612533,0.791704,0.101767
75%,38.000000,6.250000,5.615207,3.916344,1.604203,0.759596
max,50.000000,8.000000,23.232784,6.962525,4.055300,2.573360


## Initialize Double Demeaning Analysis

Now let's set up the analysis using the `DoubleDemeanAnalysis` class. This will validate our data and prepare it for the double demeaning procedure.


In [4]:
print("🔬 Initializing Double Demeaning Analysis...")
print("=" * 60)

analysis = DoubleDemeanAnalysis(
    data=df,
    unit_var='unit_id',
    time_var='time_id', 
    y_var='outcome',
    x_var='treatment',
    z_var='moderator',
    w_vars=['control']
)

print("✅ Analysis object created successfully!")
print(f"   🎯 Dependent variable: {analysis.y_var}")
print(f"   ⚡ Interaction: {analysis.x_var} × {analysis.z_var}")
print(f"   🎛️ Controls: {analysis.w_vars}")


INFO - Panel Data Summary:
INFO -   Total observations: 400
INFO -   Number of units: 50
INFO -   Time periods per unit: 8-8
INFO -   Panel type: Balanced
INFO - Data prepared: 400 observations, 50 units


🔬 Initializing Double Demeaning Analysis...
✅ Analysis object created successfully!
   🎯 Dependent variable: outcome
   ⚡ Interaction: treatment × moderator
   🎛️ Controls: ['control']


## Run the Complete Analysis

This is the main analysis step. The `run_analysis()` method will:

1. **Data preparation**: Validate panel structure and check variation
2. **Grand mean centering**: Optional preprocessing step
3. **Double demeaning**: Create the unbiased interaction term
4. **Model estimation**: Compare standard FE vs double demeaned FE
5. **Statistical testing**: Hausman test for systematic differences


In [5]:
# Run the complete analysis with verbose output
results = analysis.run_analysis(
    center_variables=True,  # Apply grand mean centering
    run_hausman=True,      # Perform Hausman test
    verbose=True           # Show detailed output
)


INFO - STARTING DOUBLE DEMEANING ANALYSIS
INFO - STARTING DOUBLE DEMEANING ANALYSIS
INFO - Dataset: 400 observations, 4 variables
INFO - Dataset: 400 observations, 4 variables
INFO - Panel structure: unit_id (units) x time_id (time)
INFO - Panel structure: unit_id (units) x time_id (time)
INFO - Analysis: outcome ~ treatment x moderator + controls
INFO - Analysis: outcome ~ treatment x moderator + controls
INFO - Step 1: Data Preparation
INFO - Step 1: Data Preparation
INFO -   Panel: 50 units, 8-8 periods per unit (mean 8.0)
INFO -   Panel: 50 units, 8-8 periods per unit (mean 8.0)
INFO - Step 2: Grand Mean Centering
INFO - Step 2: Grand Mean Centering
INFO -   outcome: mean before = 3.61681, mean after = 0.0000000000
INFO -   outcome: mean before = 3.61681, mean after = 0.0000000000
INFO -   treatment: mean before = 2.55717, mean after = -0.0000000000
INFO -   treatment: mean before = 2.55717, mean after = -0.0000000000
INFO -   moderator: mean before = 0.85917, mean after = 0.000000

## Interpret the Results

Now let's extract and interpret the key findings from our analysis.


In [6]:
print("=" * 70)
print("RESULTS SUMMARY")
print("=" * 70)

# Access results via the typed dataclass API
comparison = results.comparison
hausman = results.hausman

print(f"\nResult type: {type(results).__name__}")
print(f"Comparison type: {type(comparison).__name__}")

# Coefficient comparison table
print("\nCOEFFICIENT COMPARISON TABLE:")
print(comparison.table.to_string(index=False))

# Highlight the interaction effect
print(f"\nKEY FINDING - Interaction coefficient difference: {comparison.interaction_difference:.6f}")

interaction_row = comparison.table[comparison.table["Variable"].str.contains("int_")]
if len(interaction_row) > 0:
    row = interaction_row.iloc[0]
    print(f"  Standard FE interaction: {row['Std_FE_Coef']:.6f} (SE: {row['Std_FE_SE']:.6f})")
    print(f"  DD FE interaction:       {row['DD_Coef']:.6f} (SE: {row['DD_SE']:.6f})")
    if abs(comparison.interaction_difference) > 0.1:
        print("  SUBSTANTIAL DIFFERENCE — standard FE may be biased.")
    else:
        print("  Small difference — both estimators appear similar.")


RESULTS SUMMARY

Result type: AnalysisResult
Comparison type: ComparisonResult

COEFFICIENT COMPARISON TABLE:
               Variable  Std_FE_Coef  Std_FE_SE  DD_Coef    DD_SE  Difference
              treatment     1.229217   0.049972 1.225278 0.074700    0.003939
              moderator     1.832722   0.071099 1.767363 0.143131    0.065359
int_treatment_moderator     0.508253   0.041595 0.476582 0.083950    0.031671
                control     0.322545   0.042245 0.326583 0.058093   -0.004039

KEY FINDING - Interaction coefficient difference: 0.031671
  Standard FE interaction: 0.508253 (SE: 0.041595)
  DD FE interaction:       0.476582 (SE: 0.083950)
  Small difference — both estimators appear similar.


In [7]:
print("HAUSMAN TEST RESULTS:")
print("=" * 40)

if hausman is not None:
    print(f"  Test statistic (chi2): {hausman.statistic:.4f}")
    print(f"  Degrees of freedom:   {hausman.degrees_of_freedom}")
    print(f"  P-value:              {hausman.p_value:.4f}")
    print(f"  Positive definite:    {hausman.positive_definite}")
    print(f"  Conclusion:           {hausman.conclusion}")

    if hausman.p_value < 0.05:
        print("\n  REJECT H0 (p < 0.05). Evidence of systematic bias.")
        print("  Prefer the double-demeaned estimator.")
    else:
        print("\n  FAIL TO REJECT H0 (p >= 0.05). No systematic differences.")
        print("  Both estimators appear consistent; standard FE is more efficient.")
else:
    print("  Hausman test was not run (run_hausman=False).")


HAUSMAN TEST RESULTS:
  Test statistic (chi2): 0.5128
  Degrees of freedom:   4
  P-value:              0.9722
  Positive definite:    True
  Conclusion:           NO_SYSTEMATIC_BIAS

  FAIL TO REJECT H0 (p >= 0.05). No systematic differences.
  Both estimators appear consistent; standard FE is more efficient.


## Conclusion

This notebook demonstrated the complete workflow for double demeaning analysis:

1. **✅ Data Preparation**: Generated synthetic panel data with realistic structure
2. **✅ Analysis Setup**: Initialized the DoubleDemeanAnalysis class
3. **✅ Model Estimation**: Compared standard FE vs double demeaned FE models
4. **✅ Statistical Testing**: Performed Hausman test for systematic differences
5. **✅ Results Interpretation**: Provided clear guidance on which estimator to prefer

### Key Takeaways:

- **Double demeaning** provides an unbiased within-unit interaction estimator when there is unobserved effect heterogeneity
- **Hausman test** helps determine whether standard FE estimates are biased
- **Both approaches** should be reported for transparency and robustness

### For Your Research:

1. Replace the synthetic data with your real dataset
2. Adjust variable names to match your data
3. Interpret results in the context of your research question
4. Consider theoretical justifications for the interaction effects

**Happy analyzing! 📊🔬**
